# Oil Risk Premia & FX Strategy — Final Backtest
## Net of Transaction Costs (Base Case: 0.05% Round-Trip)

---

### Overview

This notebook implements and evaluates a long-short currency futures strategy
grounded in oil market theory, following the framework of
**Bouchouev & Zuo (2020): *Oil Risk Premia under Changing Regimes***.

The central hypothesis is that oil-exporting countries benefit when physical
oil markets are tight (backwardation), and suffer when inventories are ample
(contango). This asymmetry should manifest in the relative performance of
oil-exporter vs oil-importer currencies, creating an exploitable signal.

---

### Strategy Design

**Universe:** 8 currencies across oil exporters and importers:

| Bin | Countries | Currencies |
|---|---|---|
| **Top bin (exporters)** | Norway, Canada/Mexico*, Colombia | NOK, CAD/MXN*, COP |
| **Bottom bin (importers)** | India, Korea, Japan/ZAR* | INR, KRW, JPY/ZAR* |

*Composition varies by year based on annual net oil export rankings.*

**Signal:** Derived from the Brent crude futures term structure (CO3 − CO12).

| Carry = CO3 − CO12 | Regime | Trade |
|---|---|---|
| > 0 | **Backwardation** — physical oil scarce | Long top-bin FX, Short bottom-bin FX |
| < 0 | **Contango** — inventories ample | Long bottom-bin FX, Short top-bin FX |

**Two signal variants tested:**
- **CARRY:** Reacts to the *level* of the carry spread (positive/negative)
- **CARRY-MOM:** Reacts to the *direction* of carry relative to its 4-week moving average

**Position sizing:** Equal weighting within each bin (1/3 per currency).

---

### Signal Timing — Thursday-to-Friday Implementation

A key methodological feature of this implementation is the use of
**Thursday closing prices** to construct the carry signal, with trades
executed at **Friday closing prices**:

```
Signal observed:  Thursday close  (CO3 and CO12 daily data)
Trade executed:   Friday close    (FX futures weekly close)
```

This ensures the signal is always formed from information available
**before** the execution price is determined — making the strategy
empirically realistic. Using the same closing price to both observe
a signal and execute a trade is operationally impossible in practice.
The Thursday-to-Friday approach provides a full trading day between
observation and execution.

Five Thursdays in the sample fall on public holidays (Christmas/New Year).
For these weeks, Wednesday's close is used as the signal observation day.

---

### Transaction Cost Treatment

A fixed round-trip cost of **0.05% per signal flip** is applied throughout,
following the standard convention in the currency risk premia literature
(Lustig, Roussanov & Verdelhan 2011; Menkhoff et al. 2012;
Asness, Moskowitz & Pedersen 2013). A cost is charged only when the signal
changes direction, as that is the only point at which the full portfolio
must be traded.

---

### COP (Colombia) Note

Colombia is the second-largest net oil exporter in the ranking every year.
No exchange-traded COP futures exist; the **1-Month NDF Outright
(CLN+1M BGN Curncy)** is used as a substitute. The NDF is quoted as COP
per USD and inverted to USD per COP to match the master file convention.
COP exhibits long-run structural depreciation (Colombian inflation differential
vs US), correctly priced into the NDF via interest rate differentials.
Strategy returns are driven by weekly directional moves, not price levels.

---

**Sample period:** 2007-06-22 to 2025-12-26
(bin assignments available through 2025; data trimmed accordingly)

## 0. Imports & Settings

Standard scientific Python stack. `seaborn` for the correlation heatmap;
`scipy.stats` for the normal distribution overlay on return histograms.

Key parameters set here:
- `PATH_SIGNAL`: points to `Signal_Carry_CarryMom_Thursday.csv` — the
  Thursday-based signal file constructed from daily CO3/CO12 data
- `TC_RATE = 0.0005`: 0.05% fixed round-trip cost per signal flip
- `BACKTEST_END = '2025-12-31'`: trims data to avoid trading on 2026 for
  which bin assignments are not yet available

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.dpi': 130,
    'figure.facecolor': 'white',
    'axes.facecolor': '#f9f9f9',
    'axes.grid': True,
    'grid.alpha': 0.4,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'font.size': 11,
})

# ── File paths ────────────────────────────────────────────────────────────
PATH_FX     = 'fx_data_final.csv'   # 8 currencies incl. COP NDF
PATH_SIGNAL = 'Signal_Carry_CarryMom_Thursday.csv'         # Brent CO3/CO12 carry signals

# ── Backtest parameters ───────────────────────────────────────────────────
BACKTEST_END = '2025-12-31'   # trim to avoid using undefined 2026 bins
FREQ         = 52             # weekly -> annual conversion factor

# ── Transaction cost ─────────────────────────────────────────────────────
# Fixed round-trip cost applied once per signal flip to the full portfolio.
# 0.05% (5 bps) is the standard blended EM assumption in the FX carry literature.
TC_RATE = 0.0005  # 0.05% round-trip per signal flip


## 1. Load & Trim Data

Two input files:
1. **Master FX file** (`Master_FX_Futures_with_COP.csv`) — weekly Friday
   closing prices for 8 currencies, all transformed to USD per 1 unit of
   foreign currency
2. **Thursday signal file** (`Signal_Carry_CarryMom_Thursday.csv`) — carry
   signals computed from **Thursday closing prices** of CO3 and CO12, indexed
   on Friday dates for alignment with the FX data

Both are trimmed to end-2025 and aligned on their common Friday date index.

**Signal preview note:** The first three rows of `CARRY_MA4` appear as `NaN`
because the 4-week rolling mean requires at least 4 observations. These rows
are excluded from the backtest automatically when signals are applied.

In [ ]:
fx  = pd.read_csv(PATH_FX,     parse_dates=['Date'], index_col='Date')
sig = pd.read_csv(PATH_SIGNAL, parse_dates=['Date'], index_col='Date')

# Trim to backtest window
fx  = fx.loc[:BACKTEST_END]
sig = sig.loc[:BACKTEST_END]

# Align on common dates (should already match — this is a safety step)
common_idx = fx.index.intersection(sig.index)
fx  = fx.loc[common_idx]
sig = sig.loc[common_idx]

print('FX master   :', fx.shape,
      '| range:', fx.index.min().date(), '->', fx.index.max().date())
print('Signal file :', sig.shape,
      '| range:', sig.index.min().date(), '->', sig.index.max().date())
print('\nCurrencies :', list(fx.columns))
print('\nNaN check (all should be 0):')
print(fx.isna().sum().to_string())
print('\nFX data preview (USD per 1 unit of foreign currency):')
display(fx.head(3).round(6))
print('\nSignal data preview:')
display(sig[['CO3','CO12','CARRY','CARRY_MA4',
             'SIGNAL_CARRY','SIGNAL_CARRY_MOM']].head(3).round(3))


## 2. Annual Bin Assignments

Countries are ranked each year by **net annual oil exports** sourced from
EIA/IEA data. The top bin (net exporters) goes long when the carry signal
is positive; the bottom bin (net importers) goes short. Bins are fixed for
the full calendar year and refreshed each January — consistent with the
timing of annual energy statistics publication.

**Key observations:**
- **Norway (NOK)** is in the top bin every year — a permanent anchor
- **Colombia (COP)** is in the top bin every year — second-largest exporter
- **Mexico (MXN)** rotates out of the top bin after 2010 as Canadian oil
  sands production overtakes Mexican output, and re-enters the bottom bin
  in 2020–2022 when Mexico became a net importer
- **South Africa (ZAR)** rotates into the bottom bin in selected years
  when Japan is displaced — ZAR is an energy importer despite being a
  commodity exporter

**Ticker mapping:**
Norway=NOK, Canada=CAD, Mexico=MXN, Colombia=COP (NDF),
India=INR, Korea=KRW, Japan=JPY, South Africa=ZAR

In [ ]:
BIN_ASSIGNMENTS = {
    2007: {'top': ['NOK','MXN','COP'], 'bottom': ['INR','KRW','JPY']},
    2008: {'top': ['NOK','MXN','COP'], 'bottom': ['INR','KRW','JPY']},
    2009: {'top': ['NOK','MXN','COP'], 'bottom': ['INR','KRW','ZAR']},
    2010: {'top': ['NOK','MXN','COP'], 'bottom': ['INR','KRW','JPY']},
    2011: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2012: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2013: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2014: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2015: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','ZAR']},
    2016: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','ZAR']},
    2017: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2018: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','ZAR']},
    2019: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2020: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','MXN']},
    2021: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','MXN']},
    2022: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','MXN']},
    2023: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2024: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
    2025: {'top': ['NOK','CAD','COP'], 'bottom': ['INR','KRW','JPY']},
}

top_bins    = {}
bottom_bins = {}
for date in fx.index:
    yr = date.year
    if yr not in BIN_ASSIGNMENTS:
        raise ValueError(f'No bin assignment for year {yr}. '
                         f'Add it to BIN_ASSIGNMENTS or trim the data.')
    top_bins[date]    = BIN_ASSIGNMENTS[yr]['top']
    bottom_bins[date] = BIN_ASSIGNMENTS[yr]['bottom']

print('Bin assignments by year (same-year rankings used):')
print(f'  {"Year":<6} {"Top Bin (exporters)":<32} Bottom Bin (importers)')
print('  ' + '-'*72)
prev = None
for yr, b in sorted(BIN_ASSIGNMENTS.items()):
    note = '  <- change' if b != prev else ''
    print(f'  {yr:<6} {str(b["top"]):<32} {b["bottom"]}{note}')
    prev = b


## 3. Data Exploration

Before running the strategy, we examine the raw data to understand the
statistical properties of the currencies and signals. This section serves
both as a data quality check and as contextual evidence for interpreting
the strategy results.

### 3a. FX Price Levels Over Time

All prices are in **USD per 1 unit of local currency** (post-transformation).
A declining price means the currency is weakening against the USD.

Green shading marks periods when each currency is in the **top bin**
(oil exporter — typically held long during backwardation).

**Notable observations:**
- **NOK and MXN** show sharp declines during the 2014–2016 oil price crash,
  consistent with their oil-exporter status — they weaken when oil revenues fall
- **COP** shows a persistent long-run downtrend driven by Colombia's higher
  inflation differential vs the US. This is correctly priced into the NDF
  via interest rate differentials and does not bias the strategy —
  weekly returns, not price levels, drive performance
- **JPY** appreciates sharply during risk-off episodes (2008 GFC, 2020 COVID),
  consistent with its safe-haven status — these are periods when the strategy
  is typically short JPY (bottom bin in backwardation), so appreciation hurts
- **ZAR** is the most volatile currency in the universe (16.5% ann. vol),
  reflecting South Africa's high EM risk premium and commodity exposure
- **CAD and NOK** co-move strongly, especially during oil price cycles,
  confirming their shared commodity currency character

In [ ]:
fx_ret = np.log(fx / fx.shift(1)).dropna(how='all')

fig, axes = plt.subplots(4, 2, figsize=(16, 13), sharex=True)
axes_flat = axes.flatten()
colors = ['steelblue','darkorange','green','crimson',
          'purple','brown','teal','goldenrod']
ccys   = list(fx.columns)

for i, (ccy, color) in enumerate(zip(ccys, colors)):
    ax = axes_flat[i]
    ax.plot(fx.index, fx[ccy], lw=1.5, color=color)
    in_top = pd.Series(
        [ccy in top_bins.get(d, []) for d in fx.index],
        index=fx.index
    )
    ymin = fx[ccy].min() * 0.97
    ymax = fx[ccy].max() * 1.03
    ax.fill_between(fx.index, ymin, ymax,
                    where=in_top, alpha=0.09, color='green')
    bin_lbl = 'TOP BIN' if ccy in ['NOK','CAD','MXN','COP'] else 'BOTTOM BIN'
    ax.set_title(f'{ccy}  [{bin_lbl}]',
                 fontweight='bold', fontsize=11, color=color)
    ax.set_ylabel('USD per unit')
    ax.xaxis.set_major_locator(mdates.YearLocator(4))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

fig.suptitle(
    'FX Futures / NDF Price Levels  (USD per 1 unit of local currency)\n'
    'Green shading = periods in top bin (oil exporter)',
    fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()
plt.show()


### 3b. FX Return Correlation Heatmap

Pairwise correlation of weekly log-returns, with currencies grouped by bin.
The thick black line marks the bin boundary (top bin above, bottom bin below).

**Within the top bin (exporters) — positive correlations:**
All four oil-exporter currencies are positively correlated, reflecting their
shared sensitivity to global oil prices and risk appetite:
- NOK–CAD: 0.628 — highest pair; both are liquid G10 commodity currencies
- CAD–MXN: 0.557, NOK–MXN: 0.539 — strong LatAm–Scandinavian oil linkage
- COP–MXN: 0.461, NOK–COP: 0.435, CAD–COP: 0.390 — COP confirms its
  oil-exporter character despite being an NDF from an emerging market

**Within the bottom bin (importers) — near-zero correlations:**
INR, KRW, and JPY are nearly uncorrelated with each other (all below 0.09).
Each is driven by distinct domestic factors — India by current account and
inflation, Korea by tech export cycles, Japan by risk-off flows and BOJ policy.
This near-zero within-bin correlation means the short leg is effectively
three independent positions — far better diversified than the long leg.

**Implication for strategy design:**
The top bin has effective diversification of approximately 2.6 currencies
(vs 3 nominal, due to correlation), while the bottom bin has effective
diversification of nearly 3.0 currencies. This asymmetry slightly concentrates
risk on the long side — a known feature of oil-exporter currency portfolios,
not an oversight.

In [ ]:
corr = fx_ret.corr()
top_ccys    = ['NOK','CAD','MXN','COP']
bottom_ccys = ['INR','KRW','JPY','ZAR']
ordered     = top_ccys + bottom_ccys
corr_ordered = corr.loc[ordered, ordered]

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(
    corr_ordered, annot=True, fmt='.2f', cmap='RdYlGn',
    vmin=-1, vmax=1, center=0,
    linewidths=0.5, ax=ax, annot_kws={'size': 11}
)
ax.axhline(4, color='black', lw=2.5)
ax.axvline(4, color='black', lw=2.5)
ax.text(0.7, -0.35, 'TOP BIN (exporters)',
        fontsize=10, fontweight='bold', color='darkgreen')
ax.text(5.0, -0.35, 'BOTTOM BIN (importers)',
        fontsize=10, fontweight='bold', color='darkred')
ax.set_title(
    'FX Weekly Return Correlation Matrix\n'
    '(Grouped by bin — thick line = boundary)',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

print('Top-bin internal correlations:')
for i, a in enumerate(top_ccys):
    for b in top_ccys[i+1:]:
        print(f'  {a}-{b}: {corr.loc[a,b]:.3f}')
print('\nBottom-bin internal correlations:')
for i, a in enumerate(bottom_ccys):
    for b in bottom_ccys[i+1:]:
        if a in corr.index and b in corr.index:
            print(f'  {a}-{b}: {corr.loc[a,b]:.3f}')


### 3c. Rolling 52-Week Correlation — NOK vs COP and NOK vs CAD

These rolling correlations track whether COP and CAD consistently co-move
with NOK — the core justification for placing them together in the top bin.

**NOK–COP (left panel):** Correlation averages around 0.40–0.45 and is
persistently positive throughout the sample. The correlation strengthens
during major oil price cycles: the 2008 spike, the 2014–2016 crash, and
the 2020–2022 recovery. This confirms that both currencies respond to
the same underlying oil market driver, validating COP's inclusion in the
top bin despite its emerging market status.

**NOK–CAD (right panel):** Higher and more stable correlation (~0.63 average)
reflects both being G10 commodity currencies with deep, liquid markets.
The dip during 2014–2016 (oil crash) is notable — both depreciated heavily,
but CAD was partially supported by North American macro factors, briefly
reducing the co-movement with NOK.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, (a, b), color in [
    (axes[0], ('NOK','COP'), 'steelblue'),
    (axes[1], ('NOK','CAD'), 'darkorange'),
]:
    rc = fx_ret[a].rolling(52).corr(fx_ret[b])
    ax.plot(rc.index, rc.values, lw=1.8, color=color)
    ax.axhline(rc.mean(), color='black', lw=1.2, linestyle='--',
               label=f'Full-sample mean: {rc.mean():.2f}')
    ax.axhline(0, color='black', lw=0.6, linestyle=':')
    ax.fill_between(rc.index, rc.values, rc.mean(),
                    where=rc.values > rc.mean(),
                    alpha=0.18, color='green',
                    label='Above mean')
    ax.fill_between(rc.index, rc.values, rc.mean(),
                    where=rc.values <= rc.mean(),
                    alpha=0.18, color='red',
                    label='Below mean')
    ax.set_title(f'Rolling 52-Week Correlation: {a} vs {b}',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Correlation')
    ax.set_ylim(-0.5, 1.0)
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator(3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Rolling Top-Bin Correlations (52-week window)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 3d. Brent Carry Signal — Full History

The signal is built from the Brent futures term structure using
**Thursday closing prices**: `CARRY = CO3_Thursday − CO12_Thursday`
(3rd-nearby minus 12th-nearby price). Using Thursday prices ensures the
signal is formed before Friday's FX execution price is set.

**Panel 1 — Brent futures prices:** The gap between CO3 and CO12 is the
raw input. When CO3 > CO12 (near > far), the curve is in backwardation;
when CO3 < CO12, it is in contango.

**Panel 2 — Carry spread with 4-week MA:** Key episodes in the sample:
- **2007–2008:** Deepening backwardation as demand surged to +17.8 USD/bbl
- **2008–2009:** Sharp reversal to contango as GFC crashed demand and
  inventories swelled
- **2010–2014:** Mostly backwardation as emerging market demand recovered
- **2014–2016:** Deep structural contango (shale supply surge, OPEC market
  share strategy drove inventories to record highs)
- **2020:** Brief but severe contango at COVID-19 demand collapse
- **2021–2022:** Strong backwardation as post-COVID demand recovery
  outpaced supply

**Panel 3 — Binary signals:** CARRY flips only when the spread crosses
zero; CARRY-MOM flips when carry crosses its 4-week moving average —
an earlier trigger anticipating regime transitions. Because CARRY-MOM
reacts to the *direction* of carry change rather than its level, it
fires more frequently and earlier than CARRY.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(14, 11), sharex=True)

# Panel 1: CO3 and CO12 prices
axes[0].plot(sig.index, sig['CO3'],  lw=1.5, color='steelblue',
             label='CO3 (3rd-nearby)')
axes[0].plot(sig.index, sig['CO12'], lw=1.5, color='firebrick',
             linestyle='--', label='CO12 (12th-nearby)')
axes[0].set_ylabel('Price (USD/bbl)')
axes[0].set_title('Brent Futures: Near (CO3) vs Far (CO12) Contract Prices',
                  fontweight='bold')
axes[0].legend(fontsize=10)

# Panel 2: Carry spread with MA
axes[1].plot(sig.index, sig['CARRY'],     lw=1.5, color='purple',
             label='CARRY = CO3 − CO12')
axes[1].plot(sig.index, sig['CARRY_MA4'], lw=1.5, color='orange',
             linestyle='--', label='4-week MA of CARRY')
axes[1].axhline(0, color='black', lw=0.9, linestyle=':')
axes[1].fill_between(sig.index, sig['CARRY'], 0,
                     where=sig['CARRY'] >= 0,
                     alpha=0.20, color='green', label='Backwardation')
axes[1].fill_between(sig.index, sig['CARRY'], 0,
                     where=sig['CARRY'] < 0,
                     alpha=0.20, color='red', label='Contango')
axes[1].set_ylabel('Spread (USD/bbl)')
axes[1].set_title('CARRY Spread and 4-Week Moving Average',
                  fontweight='bold')
axes[1].legend(fontsize=9, ncol=2)

# Panel 3: Signals
axes[2].step(sig.index, sig['SIGNAL_CARRY'],     lw=2.0,
             color='steelblue',  where='post', label='CARRY signal')
axes[2].step(sig.index, sig['SIGNAL_CARRY_MOM'], lw=2.0,
             color='darkorange', where='post', linestyle='--',
             label='CARRY-MOM signal')
axes[2].axhline(0, color='black', lw=0.5)
axes[2].set_yticks([-1, 1])
axes[2].set_yticklabels(['-1  (Contango → Long importers / Short exporters)',
                          '+1  (Backwardation → Long exporters / Short importers)'])
axes[2].set_title('Trading Signals', fontweight='bold')
axes[2].legend(fontsize=10)

for ax in axes:
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Brent Oil Carry Signal — Full History (2007–2025)',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


### 3e. Carry Distribution & Regime Statistics

Understanding the statistical distribution of the carry spread contextualises
when the strategy is in its strongest and weakest environments.

**From the actual data (Thursday-based signal):**
- Mean carry: **+0.87 USD/bbl** (positive — structural backwardation bias)
- Median carry: **+1.70 USD/bbl** (above mean — right-skewed distribution)
- Range: −12.49 to +17.81 USD/bbl (wide, driven by 2008 and 2014–2016 extremes)
- **Backwardation weeks: 604 / 968 = 62.4%** of the sample
- **Contango weeks: 364 / 968 = 37.6%** of the sample

The distribution is right-skewed: extreme backwardation (2008 oil spike)
is more severe than extreme contango (2009, 2015–2016), pulling the mean
above the median. The 62.4% backwardation frequency means the strategy
has a structural positive tilt toward long exporters — consistent with
the historical outperformance of oil-exporter currencies over the long run.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(sig['CARRY'], bins=60, color='steelblue',
             edgecolor='white', alpha=0.85)
axes[0].axvline(0, color='black', lw=1.5, linestyle='--',
                label='Zero (regime boundary)')
axes[0].axvline(sig['CARRY'].mean(), color='darkorange', lw=1.5,
                linestyle='--',
                label=f'Mean: {sig["CARRY"].mean():.2f} USD/bbl')
axes[0].axvline(sig['CARRY'].median(), color='green', lw=1.2,
                linestyle=':',
                label=f'Median: {sig["CARRY"].median():.2f} USD/bbl')
axes[0].set_title('Carry Distribution (CO3 − CO12)', fontweight='bold')
axes[0].set_xlabel('Carry (USD/bbl)')
axes[0].set_ylabel('Frequency')
axes[0].legend(fontsize=9)

axes[1].fill_between(sig.index, sig['CARRY'], 0,
                     where=sig['CARRY'] >= 0,
                     color='green', alpha=0.40, label='Backwardation')
axes[1].fill_between(sig.index, sig['CARRY'], 0,
                     where=sig['CARRY'] < 0,
                     color='red', alpha=0.40, label='Contango')
axes[1].axhline(0, color='black', lw=0.8)
axes[1].set_title('Carry Regime Over Time', fontweight='bold')
axes[1].set_ylabel('Carry (USD/bbl)')
axes[1].legend(fontsize=10)
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()

print('Carry spread summary statistics:')
print(sig['CARRY'].describe().round(3).to_string())
n_back = (sig['CARRY'] > 0).sum()
n_cont = (sig['CARRY'] < 0).sum()
n_tot  = len(sig)
print(f'\nBackwardation weeks : {n_back} / {n_tot} ({n_back/n_tot*100:.1f}%)')
print(f'Contango weeks      : {n_cont} / {n_tot} ({n_cont/n_tot*100:.1f}%)')


## 4. Weekly Log-Returns

Weekly log-returns are computed as `r_t = log(P_t / P_{t-1})` for each
currency. A positive return means the currency appreciated against the USD.

**All currencies show negative annualised returns** over the sample period —
this reflects broad USD strength since the GFC. This is not a concern
for the strategy: profits come from the **relative performance** of exporters
vs importers conditional on the carry regime, not from absolute currency
appreciation. The strategy is dollar-neutral by construction (equal notional
long and short), so the USD direction cancels out.

**Individual currency observations:**
- **ZAR** has the highest volatility (16.48%) and most negative return (−4.64%),
  reflecting South Africa's EM risk premium
- **COP** has −3.62% annualised return — the structural NDF depreciation
  discussed in the introduction
- **NOK** has high volatility (12.99%) and is the most oil-price-sensitive
  G10 currency, making it the top bin's most informative signal carrier
- **INR** has the lowest volatility (6.91%) — India's capital controls and
  RBI management dampen currency moves relative to the underlying fundamentals

In [ ]:
fx_ret = np.log(fx / fx.shift(1)).dropna(how='all')

ann_stats = pd.DataFrame({
    'Ann. Return (%)': (fx_ret.mean() * FREQ * 100).round(2),
    'Ann. Vol (%)':    (fx_ret.std()  * np.sqrt(FREQ) * 100).round(2),
    'Skewness':        fx_ret.skew().round(3),
    'Excess Kurtosis': fx_ret.kurt().round(3),
})
print('Individual currency return statistics (full backtest sample):')
display(ann_stats)


## 5. Transaction Cost Model

### Methodology

We apply a **fixed round-trip cost of 0.05%** per signal flip, following
the standard approach in the academic FX carry literature. This is chosen
as the base-case blended assumption because:

- G10 FX futures (NOK, CAD, JPY) typically trade at 0.01–0.02% round-trip
- EM futures (KRW, MXN) trade at 0.03–0.06% round-trip
- EM NDFs (COP) trade at 0.05–0.10% round-trip given the OTC dealer spread
- A blended average across the 6 active positions falls near 0.05%

### When a Cost Is Charged

A cost is incurred **only when the signal changes direction** (+1→−1 or
−1→+1). At that point, all 6 active positions must be reversed. For an
equal-weighted portfolio (1/3 per side):

```
cost per flip = 2 × TC_RATE
```

The factor of 2 accounts for both the long leg and the short leg.

### Why Not Per-Currency Varying Costs?

Granular bid-offer data for NDF markets (especially COP) is incomplete
in Bloomberg for the full 2007–2025 sample. A fixed blended rate is the
academically accepted approach when high-frequency liquidity data is
unavailable, and is more transparent and reproducible for readers.

### Thursday Signal and Flip Frequency

The Thursday-based signal has a different flip frequency from the original
Friday-based signal because Thursday and Friday carry values sometimes
differ enough to change the regime call:
- **CARRY:** 31 flips (1.7/year) — essentially unchanged; carry level is
  very stable between Thursday and Friday
- **CARRY-MOM:** 307 flips (16.5/year) — higher than the Friday version
  (279 flips) because Thursday carry sometimes crosses the MA when Friday
  carry does not, and vice versa, creating additional flip events at
  regime boundaries

In [ ]:
# Compute signal flip indicators
flip_carry = sig['SIGNAL_CARRY'].ne(sig['SIGNAL_CARRY'].shift()).astype(int)
flip_mom   = sig['SIGNAL_CARRY_MOM'].ne(sig['SIGNAL_CARRY_MOM'].shift()).astype(int)
flip_carry.iloc[0] = 0
flip_mom.iloc[0]   = 0

n_weeks = len(sig)
n_years = n_weeks / FREQ

carry_flips = flip_carry.sum()
mom_flips   = flip_mom.sum()

print('Signal flip analysis:')
print(f'  CARRY     : {carry_flips} flips | {carry_flips/n_weeks*100:.1f}% of weeks '
      f'| {carry_flips/n_years:.1f} flips/year')
print(f'  CARRY-MOM : {mom_flips} flips | {mom_flips/n_weeks*100:.1f}% of weeks '
      f'| {mom_flips/n_years:.1f} flips/year')

carry_cost_pa = carry_flips * 2 * TC_RATE / n_years * 100
mom_cost_pa   = mom_flips   * 2 * TC_RATE / n_years * 100
print(f'\nEstimated annual cost drag at {TC_RATE*100:.2f}% round-trip:')
print(f'  CARRY     : {carry_cost_pa:.3f}% per year '
      f'({carry_flips * 2 * TC_RATE * 100:.2f}% total over sample)')
print(f'  CARRY-MOM : {mom_cost_pa:.3f}% per year '
      f'({mom_flips * 2 * TC_RATE * 100:.2f}% total over sample)')
print(f'\nNote: CARRY-MOM incurs {mom_flips/carry_flips:.0f}x more flips than CARRY,'
      f' but starts with a much higher gross return.')


## 6. Backtesting Engine

The engine iterates over each week in the common sample and:

1. Reads the **Thursday-based signal** (+1 or −1) for that Friday date
2. Looks up the active top-bin and bottom-bin currencies for that year
3. Assigns equal weights: `+signal/3` to each top-bin currency,
   `−signal/3` to each bottom-bin currency
4. Computes the **gross weekly return** as the weighted sum of individual
   currency log-returns (measured from Friday close to Friday close)
5. Checks if the signal **flipped** from the previous week. If so,
   deducts `2 × TC_RATE` from the gross return to get the **net return**

**Timing:** The signal observed on Thursday determines the position held
from Friday close to the following Friday close. This is a clean
one-trading-day gap between signal formation and position entry.

Gross and net returns are stored separately so the impact of transaction
costs can be isolated in subsequent analysis.

In [ ]:
def run_backtest(signal_col):
    """
    Run the long-short FX futures / NDF strategy backtest.

    Parameters
    ----------
    signal_col : str
        Column in `sig` to use as signal:
        'SIGNAL_CARRY' or 'SIGNAL_CARRY_MOM'

    Returns
    -------
    pd.DataFrame with columns:
        gross_return  weekly return before transaction costs
        tc_charge     transaction cost deducted (0 if no flip)
        net_return    gross_return - tc_charge
        signal        +1 or -1 signal active that week
        flip          1 if signal changed from prior week, else 0
        top_bin       active top-bin tickers
        bottom_bin    active bottom-bin tickers
        {ccy}_weight  position weight for each of the 8 currencies
    """
    results     = []
    prev_signal = None
    common_dates = fx_ret.index.intersection(sig.index)

    for date in common_dates:
        s      = sig.loc[date, signal_col]
        top    = top_bins[date]
        bottom = bottom_bins[date]
        n_top  = len(top)
        n_bot  = len(bottom)

        # Equal-weight positions
        weights = {}
        for ccy in fx.columns:
            if ccy in top:
                weights[ccy] = s * (1.0 / n_top)
            elif ccy in bottom:
                weights[ccy] = -s * (1.0 / n_bot)
            else:
                weights[ccy] = 0.0

        # Gross return = sum(weight * log-return) across all currencies
        gross = sum(
            weights[ccy] * fx_ret.loc[date, ccy]
            for ccy in fx.columns
        )

        # Transaction cost on signal flip
        # cost = 2 * TC_RATE (long leg + short leg, each = 1 unit notional)
        flipped   = (prev_signal is not None) and (s != prev_signal)
        tc_charge = 2 * TC_RATE if flipped else 0.0
        net       = gross - tc_charge

        row = {
            'date':         date,
            'signal':       s,
            'flip':         int(flipped),
            'gross_return': gross,
            'tc_charge':    tc_charge,
            'net_return':   net,
            'top_bin':      str(top),
            'bottom_bin':   str(bottom),
        }
        for ccy in fx.columns:
            row[f'{ccy}_weight'] = weights[ccy]
        results.append(row)
        prev_signal = s

    return pd.DataFrame(results).set_index('date')


res_carry = run_backtest('SIGNAL_CARRY')
res_mom   = run_backtest('SIGNAL_CARRY_MOM')

print('Backtest complete.')
print(f'  Weeks : {len(res_carry)}')
print(f'  Range : {res_carry.index.min().date()} '
      f'to {res_carry.index.max().date()}')
print(f'  CARRY flips    : {res_carry["flip"].sum()}')
print(f'  CARRY-MOM flips: {res_mom["flip"].sum()}')
print('\nSample output (CARRY-MOM, first 5 rows):')
display(res_mom[['signal','flip','gross_return',
                  'tc_charge','net_return','top_bin']].head())


## 7. Performance Metrics

We report both **gross** (pre-cost) and **net** (post-cost) performance
for both signals. Key metrics:

- **Information Ratio (IR):** Annualised return divided by annualised
  volatility. The primary metric used in Bouchouev & Zuo (2020).
  An IR > 1.0 over a long sample is considered strong in the literature.

- **Sortino Ratio:** Like the IR but uses only downside volatility in the
  denominator. More appropriate than the IR when return distributions are
  positively skewed, because it does not penalise upside volatility.

- **Calmar Ratio:** Annualised return divided by maximum drawdown.
  Captures the risk-adjusted return from the perspective of the worst
  capital loss an investor would have experienced.

- **Win Rate:** Fraction of weeks with a positive return. Even modestly
  above 50% is meaningful for a systematic strategy.

- **Skewness / Excess Kurtosis:** Positive skewness is desirable — it means
  the tail of large gains is longer than the tail of large losses.

In [ ]:
def compute_metrics(ret_series, label='', freq=FREQ):
    """
    Compute comprehensive performance metrics for a weekly log-return series.

    Parameters
    ----------
    ret_series : pd.Series  weekly log-returns
    label      : str        name used in output
    freq       : int        periods per year (52 for weekly)

    Returns
    -------
    dict of named performance metrics
    """
    r = ret_series.dropna()
    ann_ret  = r.mean() * freq
    ann_vol  = r.std()  * np.sqrt(freq)
    ir       = ann_ret / ann_vol if ann_vol != 0 else np.nan
    downside = r[r < 0]
    down_vol = (downside.std() * np.sqrt(freq)
                if len(downside) > 1 else np.nan)
    sortino  = (ann_ret / down_vol
                if (down_vol and down_vol != 0) else np.nan)
    cum      = r.cumsum()
    dd       = cum - cum.cummax()
    max_dd   = dd.min()
    calmar   = ann_ret / abs(max_dd) if max_dd != 0 else np.nan
    return {
        'Ann. Return (%)':     round(ann_ret  * 100, 2),
        'Ann. Volatility (%)': round(ann_vol  * 100, 2),
        'Information Ratio':   round(ir,              3),
        'Sortino Ratio':       round(sortino,          3),
        'Max Drawdown (%)':    round(max_dd   * 100,  2),
        'Calmar Ratio':        round(calmar,           3),
        'Win Rate (%)':        round((r > 0).mean() * 100, 1),
        'Skewness':            round(r.skew(),         3),
        'Excess Kurtosis':     round(r.kurt(),         3),
        'Best Week (%)':       round(r.max() * 100,    2),
        'Worst Week (%)':      round(r.min() * 100,    2),
        'N Weeks':             len(r),
    }


metrics = pd.DataFrame({
    'CARRY  Gross':  compute_metrics(res_carry['gross_return']),
    'CARRY  Net':    compute_metrics(res_carry['net_return']),
    'MOM  Gross':    compute_metrics(res_mom['gross_return']),
    'MOM  Net':      compute_metrics(res_mom['net_return']),
})

start = res_carry.index.min().strftime('%Y-%m-%d')
end   = res_carry.index.max().strftime('%Y-%m-%d')

print('=' * 60)
print('       STRATEGY PERFORMANCE SUMMARY')
print(f'       {start} to {end}')
print(f'       Transaction cost: {TC_RATE*100:.2f}% round-trip per flip')
print(f'       Universe: 8 currencies including COP NDF')
print('=' * 60)
display(metrics)


### Interpreting the Metrics

**CARRY strategy:** The gross IR of **0.379** is below the conventional
threshold of 0.5 for a deployable strategy. After costs (0.17% annual drag
from 31 signal flips at 1.7/year), the net IR is **0.359** — marginal improvement in
the Thursday version vs the prior Friday version. CARRY's fundamental
problem is regime stickiness: it provides no dynamic positioning within a
regime, acting as a passive long-exporter tilt for the 62.4% of the sample
in backwardation.

**CARRY-MOM strategy:** The gross IR of **1.284** is strong by any standard.
After costs (1.65% annual drag from 307 signal flips at 16.5/year), the net IR is
**1.081** — the Thursday implementation retains a robust and implementable
edge even after accounting for the higher flip frequency that the
daily signal resolution introduces.

**Thursday implementation vs Friday baseline:**
Compared to the original Friday-based signal, the Thursday implementation
shows a modestly different CARRY-MOM performance profile. The Thursday
signal sometimes captures regime transitions one day earlier (a benefit)
but also generates additional flips at regime boundaries where Thursday
and Friday carry values straddle the MA (a cost). The net effect is a
slightly higher flip count (307 vs 279) but a numerically similar IR
(1.081 vs 1.050), suggesting the Thursday implementation is broadly
equivalent in quality while being empirically more defensible.

**Cost impact:** CARRY-MOM loses 0.203 IR points gross-to-net vs CARRY's
0.020 IR points — the 10× flip-frequency differential is real and
meaningful. However, the strategy's IR headroom (1.284 gross) means it
remains strongly profitable net. The breakeven transaction cost where
net IR would reach zero is well above 0.20% per trade.

## 8. Annual Performance Breakdown

Year-by-year analysis of net returns (post-cost at 0.05%).
The Thursday signal introduces some differences relative to the Friday
baseline in specific years where the carry crossed the MA between
Thursday and Friday, shifting when positions were entered or exited.

In [ ]:
def annual_breakdown(ret_series, label, freq=FREQ):
    """
    Year-by-year return, volatility, Information Ratio, and Sortino Ratio.
    """
    df   = ret_series.to_frame('ret')
    rows = []
    for yr, grp in df.groupby(df.index.year):
        r        = grp['ret']
        ann_ret  = r.mean() * freq
        ann_vol  = r.std()  * np.sqrt(freq)
        ir       = ann_ret / ann_vol if ann_vol > 0 else np.nan
        downside = r[r < 0]
        down_vol = (downside.std() * np.sqrt(freq)
                    if len(downside) > 1 else np.nan)
        sortino  = (ann_ret / down_vol
                    if (down_vol and down_vol != 0) else np.nan)
        rows.append({
            'Year':            yr,
            'Ann. Return (%)': round(ann_ret * 100, 2),
            'Ann. Vol (%)':    round(ann_vol * 100, 2),
            'Info Ratio':      round(ir,      3),
            'Sortino Ratio':   round(sortino, 3),
        })
    out = pd.DataFrame(rows).set_index('Year')
    print(f'Annual breakdown ({label}):')
    display(out)
    return out


ann_carry = annual_breakdown(res_carry['net_return'], 'CARRY Net')
ann_mom   = annual_breakdown(res_mom['net_return'],   'CARRY-MOM Net')


### Annual Results Commentary

**CARRY net (Thursday signal):**
The strategy has 11 negative years out of 19 — marginally worse than the
Friday version. Its cumulative positive return is concentrated in 2008
(+33.5%), the year of the historic oil spike to $147/barrel and the GFC
crash — the two-phase 2008 cycle gave the strategy large gains in
backwardation (H1) and then from the contango flip (H2). Outside of 2008,
the 18-year cumulative return is near zero — confirming the limitation
of a pure level signal.

**CARRY-MOM net (Thursday signal):**
Five negative years out of 19: 2007 (partial year, transitional regime),
2010 (−4.45%), 2012 (−1.12%), 2025 (last year, challenging for the signal),
and the occasional noisy year. Despite more negative years than the Friday
version, the strategy remains strongly positive on a cumulative basis,
delivering positive returns across diverse economic environments:
the 2008–2009 oil cycle (+12.4% and +9.0%), the 2011 recovery (+11.7%),
the 2015–2016 shale era (+13.9% and +24.4%), the COVID rebound (2020: +16.0%),
and the 2021–2024 structural backwardation period (+8.3 to +13.4%).

**The 2016 standout:** CARRY-MOM net +24.4% (IR = 4.03) is the single
best year in the sample. This was a year of deep contango throughout,
and CARRY-MOM correctly positioned short exporters / long importers for
virtually the entire year. CARRY (−1.55%) missed this by being slow
to act in periods where carry was marginally negative — illustrating
precisely why the momentum overlay adds value.

## 9. Strategy Visualisations

### 9a. Cumulative Net Returns with Regime Shading

The primary performance chart. Green background = backwardation (exporters
long, importers short); red background = contango (reversed).

**Reading the chart:**
- The CARRY line (blue) is relatively flat outside of the 2008 episode,
  confirming its dependence on a single extreme year
- The CARRY-MOM line (orange dashed) climbs more steadily, with
  contributions from many years rather than one dominant episode
- The widening gap from 2011 onward shows CARRY-MOM consistently
  compounding ahead of CARRY across the full range of market environments

**2016 — the most informative year:**
In 2016, the oil market was in contango for the full year.
CARRY-MOM correctly positioned short exporters and earned +24.4%.
CARRY earned only −1.55% in the same year — both signals saw the same
carry data, but CARRY-MOM acted on the downward trend in carry from
early in the year, while CARRY only reacted when carry crossed zero.
This single year illustrates the momentum overlay's core value proposition.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

cum_carry = res_carry['net_return'].cumsum() * 100
cum_mom   = res_mom['net_return'].cumsum()   * 100

ax.plot(cum_carry.index, cum_carry.values, lw=2.2,
        color='steelblue', label='CARRY (net)')
ax.plot(cum_mom.index,   cum_mom.values,   lw=2.2,
        color='darkorange', linestyle='--', label='CARRY-MOM (net)')
ax.axhline(0, color='black', lw=0.8, linestyle=':')

# Regime shading from CARRY signal
s    = sig['SIGNAL_CARRY'].reindex(res_carry.index)
grps = s.ne(s.shift()).cumsum()
for _, grp in res_carry.groupby(grps):
    clr = 'green' if s.loc[grp.index[0]] == 1 else 'red'
    ax.axvspan(grp.index[0], grp.index[-1], alpha=0.055, color=clr)

ax.set_title(
    f'Cumulative Net Log-Returns (after {TC_RATE*100:.2f}% round-trip cost)\n'
    'Green = backwardation, Red = contango',
    fontsize=13, fontweight='bold'
)
ax.set_ylabel('Cumulative Net Log-Return (%)')
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()


### 9b. Gross vs Net — Impact of Transaction Costs

This chart isolates the effect of the 0.05% transaction cost assumption.
The shaded red area between gross and net lines represents the
cumulative cost paid to date.

**CARRY:** The gross and net lines are nearly indistinguishable — the
31 signal flips over 19 years cost only ~0.17% per year (total ~3.1%
over the full sample). The step-wise nature of the cost drag reflects
the infrequent, large-event nature of CARRY's regime changes.

**CARRY-MOM:** The gap widens steadily over time as 307 flips accumulate
to ~1.65% per year (total ~30.7% over the full sample). The drag is
significant but does not change the qualitative conclusion — CARRY-MOM
net still comfortably outperforms CARRY gross. The comparison confirms
that CARRY-MOM's alpha is genuine and not just an artifact of ignoring
the cost of frequent rebalancing.

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

for ax, res, title, cg, cn in [
    (axes[0], res_carry, 'CARRY Strategy',
     'steelblue', 'darkblue'),
    (axes[1], res_mom,   'CARRY-MOM Strategy',
     'darkorange', 'saddlebrown'),
]:
    cum_g = res['gross_return'].cumsum() * 100
    cum_n = res['net_return'].cumsum()   * 100
    ax.plot(cum_g.index, cum_g.values, lw=2, color=cg,
            linestyle='--', label='Gross (no costs)', alpha=0.85)
    ax.plot(cum_n.index, cum_n.values, lw=2, color=cn,
            label=f'Net ({TC_RATE*100:.2f}% per flip)')
    ax.fill_between(cum_g.index, cum_g.values, cum_n.values,
                    alpha=0.15, color='crimson',
                    label='Transaction cost drag')
    ax.axhline(0, color='black', lw=0.8, linestyle=':')
    ax.set_title(f'{title}: Gross vs Net Cumulative Returns',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Cumulative Log-Return (%)')
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.tight_layout()
plt.show()


### 9c. Cumulative Transaction Cost Drag Over Time

The running total of all transaction costs paid, shown as a positive number
(the amount subtracted from gross returns). This makes the cost explicitly
visible rather than embedding it in the net return curve.

**CARRY:** Accumulates slowly and in discrete steps — each step is one
signal flip, and with only 31 flips in 19 years the steps are large and
infrequent. Total cost over the sample: approximately **0.062%** (~3.1% annualised × 19 years = ~3.1% total, noting only 31 flips).

**CARRY-MOM:** Accumulates almost continuously, reflecting the ~16.5 flips
per year under the Thursday signal. The steeper sections correspond to
periods of volatile carry trends (2008–2009, 2014–2016, 2020) where the
MA crossings are frequent. Total cost over the sample: approximately
**30.7%** (~1.65% per year × 18.6 years) — a real and meaningful drag that is fully reflected in the
net return and IR figures reported throughout this notebook.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, res, label, color in [
    (axes[0], res_carry, 'CARRY',     'steelblue'),
    (axes[1], res_mom,   'CARRY-MOM', 'darkorange'),
]:
    cum_cost = res['tc_charge'].cumsum() * 100
    total    = cum_cost.iloc[-1]
    ax.plot(cum_cost.index, cum_cost.values,
            lw=1.8, color=color)
    ax.fill_between(cum_cost.index, cum_cost.values, 0,
                    alpha=0.2, color=color)
    ax.set_title(f'{label}: Cumulative Cost Paid\n'
                 f'(Total over sample: {total:.2f}%)',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Cumulative TC (%)')
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle(f'Cumulative Transaction Cost Drag ({TC_RATE*100:.2f}% per flip)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 9d. Drawdown

Drawdown measures the decline from the running peak of cumulative returns.
It answers: *what is the worst loss an investor would have experienced
from peak to trough at any point in the sample?*

**CARRY net:** Maximum drawdown of **−16.9%**, occurring during 2017–2019
when the carry signal was persistently stuck in backwardation while the
exporter currencies underperformed. The drawdown is deeper than in the
Friday version (−18.6%) because the Thursday signal occasionally exits
bad positions a day earlier, limiting some of the worst weeks.

**CARRY-MOM net:** Maximum drawdown of **−15.4%**. This is notably larger
than the Friday version (−9.1%), reflecting years where the Thursday MOM
signal triggered additional flips that increased cost drag in already
challenging periods (e.g. 2010, 2012). The Calmar ratio of 0.570
indicates that returns per unit of drawdown risk remain positive
but are lower than the Friday implementation.

**Key takeaway:** While CARRY-MOM's max drawdown is larger than expected
from the Friday version, the strategy still generates a positive Calmar
ratio and a net IR above 1.0, confirming the alpha is genuine.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for res, label, color in [
    (res_carry, 'CARRY (net)',     'steelblue'),
    (res_mom,   'CARRY-MOM (net)', 'darkorange'),
]:
    cum = res['net_return'].cumsum()
    dd  = (cum - cum.cummax()) * 100
    ax.plot(dd.index, dd.values, lw=1.8, color=color, label=label)
    ax.fill_between(dd.index, dd.values, 0, alpha=0.12, color=color)

ax.axhline(0, color='black', lw=0.8)
ax.set_title('Drawdown (%) — Net Returns After Transaction Costs',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Drawdown (%)')
ax.legend(fontsize=11)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()


### 9e. Annual Net Returns Bar Chart

Year-by-year net returns for both strategies. Blue/orange bars are positive
years; salmon bars are negative. The annotations show the exact return
for each year.

**CARRY:** Broadly alternating between positive and negative years with
high dispersion. The 2008 bar (+33.5%) dominates the chart. Without it,
the cumulative return would be approximately flat. This illustrates the
core weakness of a regime-level signal — it is a passive bet that
delivers when the regime happens to match the currency response.

**CARRY-MOM:** Five negative years out of 19 (2007, 2010, 2012, and to
a minor extent 2025). The losses are generally modest (the worst is 2010
at −4.45%), while the positive years span a wide range of environments.
The 2016 bar (+24.4%) is the standout — a year of deep, persistent
contango where CARRY-MOM correctly positioned short exporters.
The width and frequency of positive bars confirms genuine persistence
of the CARRY-MOM edge rather than concentration in a few lucky years.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)

for ax, ann_df, label, color in [
    (axes[0], ann_carry, f'CARRY Net ({TC_RATE*100:.2f}%)',     'steelblue'),
    (axes[1], ann_mom,   f'CARRY-MOM Net ({TC_RATE*100:.2f}%)', 'darkorange'),
]:
    vals   = ann_df['Ann. Return (%)']
    colors = [color if v >= 0 else 'salmon' for v in vals]
    bars   = ax.bar(ann_df.index, vals, color=colors,
                    edgecolor='white', linewidth=0.5)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(label, fontsize=12, fontweight='bold')
    ax.set_xlabel('Year')
    ax.set_ylabel('Ann. Net Return (%)')
    ax.set_xticks(ann_df.index)
    ax.set_xticklabels(ann_df.index, rotation=45, ha='right', fontsize=8)
    for bar, val in zip(bars, vals):
        ypos = bar.get_height() + 0.35 if val >= 0 else bar.get_height() - 1.8
        ax.text(bar.get_x() + bar.get_width()/2, ypos,
                f'{val:.1f}', ha='center', fontsize=7.5)

plt.suptitle('Annual Net Returns (after transaction costs)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 9f. Rolling 52-Week Information Ratio

The rolling IR shows whether the strategy's edge is consistent over time
or concentrated in a few years. A strategy with genuine, stable alpha
should spend most of its time above the IR = 0 line.

**CARRY net (blue):** Spends significant time below −0.5 in 2011, 2017,
and parts of 2022–2024 — prolonged stretches of underperformance. Its
positive full-sample IR is largely driven by the exceptional 2008 episode
which takes approximately 52 weeks to roll out of the window after 2009.

**CARRY-MOM net (orange):** Spends the majority of the sample above zero
and frequently above the +0.5 reference line. The dips below zero
(notably 2010 and 2012 where the Thursday MOM signal was noisier than
the Friday version) are brief and recover quickly. This pattern — mostly
above zero with modest temporary dips — is the hallmark of a strategy
with genuine, persistent alpha rather than a lucky single-year outlier.
The IR consistently above +1.0 for extended periods (2013–2016, 2019–2022)
is particularly compelling evidence of robustness.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))

for res, label, color in [
    (res_carry, 'CARRY (net)',     'steelblue'),
    (res_mom,   'CARRY-MOM (net)', 'darkorange'),
]:
    r       = res['net_return']
    roll_ir = (
        (r.rolling(52).mean() * FREQ) /
        (r.rolling(52).std()  * np.sqrt(FREQ))
    )
    ax.plot(roll_ir.index, roll_ir.values,
            lw=1.8, color=color, label=label)

ax.axhline( 0.0, color='black', lw=0.9)
ax.axhline( 1.0, color='green', lw=0.9, linestyle='--',
            alpha=0.6, label='IR = +1.0')
ax.axhline( 0.5, color='green', lw=0.8, linestyle=':',
            alpha=0.5, label='IR = +0.5')
ax.axhline(-0.5, color='red',   lw=0.8, linestyle=':',
            alpha=0.5, label='IR = −0.5')
ax.set_title('Rolling 52-Week Information Ratio (Net Returns)',
             fontsize=13, fontweight='bold')
ax.set_ylabel('Information Ratio')
ax.legend(fontsize=10, ncol=3)
ax.xaxis.set_major_locator(mdates.YearLocator(2))
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()


### 9g. Return Distribution with Normal Overlay

Weekly net return distributions compared against a fitted normal distribution
(black curve). Deviations from the normal reveal the strategy's tail behaviour.

**Key findings from the actual data:**

- Both strategies exhibit **positive skewness** (CARRY: 0.477, MOM: 0.420)
  — the right tail (large gains) is longer than the left tail (large losses).
  This is desirable for a long-short strategy and suggests the strategy
  limits downside exposure more effectively than it captures upside gains grow.

- **Excess kurtosis** of ~2.97 (CARRY) and ~3.03 (MOM) indicates fat tails
  relative to a normal distribution — extreme weeks occur more often than
  the normal would predict. Both values are lower than many equity and
  carry strategies (often kurtosis > 5), suggesting relatively well-behaved
  return distributions that are not dominated by one or two outlier events.

- **Best and worst weeks are identical** across strategies (+7.19% and
  −4.25%), confirming that the extreme events are driven by the same
  underlying FX market shocks regardless of which signal was active.

- The **normal overlay is systematically too thin** in both tails and
  too tall in the centre — the classic leptokurtic pattern. This is
  expected for weekly FX returns and does not indicate a strategy problem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, label, color in [
    (axes[0], res_carry, 'CARRY (net)',     'steelblue'),
    (axes[1], res_mom,   'CARRY-MOM (net)', 'darkorange'),
]:
    r = res['net_return'].dropna() * 100
    ax.hist(r, bins=60, density=True, color=color,
            alpha=0.60, edgecolor='white', label='Empirical')
    mu, sd = r.mean(), r.std()
    x = np.linspace(r.min(), r.max(), 300)
    ax.plot(x, stats.norm.pdf(x, mu, sd), 'k-', lw=2,
            label=f'Normal(μ={mu:.3f}, σ={sd:.3f})')
    ax.axvline(0, color='black', lw=0.8, linestyle=':')
    ax.set_title(f'{label}: Weekly Return Distribution',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Weekly Net Log-Return (%)')
    ax.set_ylabel('Density')
    ax.legend(fontsize=9)
    ax.text(
        0.97, 0.95,
        f'Mean:  {mu:.3f}%\nVol:    {sd:.3f}%\n'
        f'Skew:  {r.skew():.2f}\nKurt:  {r.kurt():.2f}',
        transform=ax.transAxes, ha='right', va='top', fontsize=9,
        bbox=dict(boxstyle='round', facecolor='white', alpha=0.85)
    )

plt.suptitle('Weekly Net Return Distributions (with Normal Overlay)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 9h. Regime Contribution Analysis

This chart decomposes cumulative net returns into the portion earned during
**backwardation weeks** (signal = +1, long exporters) vs
**contango weeks** (signal = −1, long importers). The black dashed line
shows the combined total for reference.

**CARRY net:** Almost entirely driven by backwardation weeks. The contango
contribution is negative over the full sample. This confirms CARRY is
essentially a passive long-exporter tilt — it profits when oil markets
are tight but loses money in contango. The signal provides no value in
the contango regime because it simply reverses the position rather than
timing the reversal.

**CARRY-MOM net:** Generates positive cumulative returns from **both**
backwardation and contango weeks. This is the key distinguishing feature:
the momentum overlay correctly positions the strategy in both regimes,
not just one. In backwardation, it goes long exporters; in contango,
it goes long importers — and does so with better timing than CARRY,
entering regimes earlier (at the MA crossing) and potentially exiting
before the carry fully reverses. Earning from both regimes is strong
evidence of genuine alpha rather than a one-sided regime bet.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, label in [
    (axes[0], res_carry, 'CARRY (net)'),
    (axes[1], res_mom,   'CARRY-MOM (net)'),
]:
    regime = res['signal']
    net_r  = res['net_return']

    back = net_r[regime ==  1].cumsum() * 100
    cont = net_r[regime == -1].cumsum() * 100
    back = back.reindex(net_r.index).ffill().fillna(0)
    cont = cont.reindex(net_r.index).ffill().fillna(0)

    ax.plot(back.index, back.values, lw=2, color='green',
            label=f'Backwardation weeks (total: {back.iloc[-1]:.1f}%)')
    ax.plot(cont.index, cont.values, lw=2, color='red',
            label=f'Contango weeks (total: {cont.iloc[-1]:.1f}%)')
    ax.plot(net_r.cumsum().mul(100).index,
            net_r.cumsum().mul(100).values,
            lw=1.2, color='black', linestyle='--',
            alpha=0.5, label='Total (combined)')
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{label}: Cumulative Return by Regime',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Cumulative Net Return (%)')
    ax.legend(fontsize=9)
    ax.xaxis.set_major_locator(mdates.YearLocator(3))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))

plt.suptitle('Regime Contribution: Backwardation vs Contango Weeks',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 9i. Per-Currency Total Return Contribution

Each bar shows the total contribution (in cumulative log-return percentage)
of a single currency to the strategy's total net return over the full sample.
Bars are sorted from highest to lowest contribution within each strategy.

**Positive bar (blue):** Currency added value on balance — its position
(long or short depending on bin and signal) generated positive returns
more often than negative.

**Negative bar (salmon):** Currency detracted — its structural behaviour
worked against the signal on balance.

**Currency-specific interpretation:**
- **NOK** typically leads the positive contributors given its high vol and
  permanent top-bin status — it is the most oil-price-sensitive currency
- **KRW** is typically the strongest bottom-bin contributor — Korea's
  near-total oil import dependence makes KRW highly reactive to oil regimes
- **COP** contribution reflects competition between its oil-exporter beta
  (positive in backwardation) and its structural NDF depreciation (negative
  drag when held long). The net contribution validates its inclusion
- **MXN** may appear as a detractor in some periods due to its bin rotation
  (from top bin to bottom bin in 2020–2022), which creates noisy transitions

Note: this chart measures total portfolio contribution, not the Jackknife
contribution (Section 11). A currency can have a large contribution here
but a small Jackknife score if other currencies compensate for its absence,
and vice versa.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, res, label in [
    (axes[0], res_carry, 'CARRY (net)'),
    (axes[1], res_mom,   'CARRY-MOM (net)'),
]:
    contrib = {}
    for ccy in fx.columns:
        w = res[f'{ccy}_weight']
        r = fx_ret.reindex(res.index)[ccy]
        contrib[ccy] = (w * r).sum() * 100

    contrib_s = pd.Series(contrib).sort_values()
    colors_c  = ['steelblue' if v >= 0 else 'salmon'
                 for v in contrib_s.values]
    bars = ax.bar(range(len(contrib_s)), contrib_s.values,
                  color=colors_c, edgecolor='white')
    ax.set_xticks(range(len(contrib_s)))
    ax.set_xticklabels(contrib_s.index, fontsize=11)
    ax.axhline(0, color='black', lw=0.8)
    ax.set_title(f'{label}: Total Contribution by Currency',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('Total Net Return Contribution (%)')
    for i, (ccy, v) in enumerate(contrib_s.items()):
        ypos = v + 0.25 if v >= 0 else v - 1.0
        ax.text(i, ypos, f'{v:.1f}%', ha='center', fontsize=9)

plt.suptitle('Per-Currency Contribution to Total Net Strategy Return',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


### 9j. Signal Flip Frequency Over Time

Rolling 52-week count of signal direction changes. This directly determines
the transaction cost drag shown in Section 9c.

**CARRY (blue):** Extremely stable at approximately 1.7 flips per year
(32 total over the sample). The carry level rarely crosses zero —
when it does, it typically stays on one side for months or years.
Spikes in the rolling count correspond to the volatile 2008–2009 period
when the carry swung from deep backwardation to deep contango.

**CARRY-MOM (orange):** Averages 16.5 flips per year (308 total) under
the Thursday signal — noticeably higher than the Friday version (15.0/year)
because Thursday carry sometimes crosses the 4-week MA when Friday carry
does not, adding flip events at regime boundaries. Peaks at 2008–2009,
2014–2016, and 2020 correspond to periods of volatile carry trends —
exactly when the momentum overlay is most active and most valuable.
The elevated flip frequency in these periods also coincides with the
strategy's strongest performance years, confirming that higher turnover
is a feature of high-signal-strength environments rather than noise.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

for ax, col, label, color in [
    (axes[0], 'SIGNAL_CARRY',     'CARRY',     'steelblue'),
    (axes[1], 'SIGNAL_CARRY_MOM', 'CARRY-MOM', 'darkorange'),
]:
    flips      = sig[col].ne(sig[col].shift()).astype(int)
    roll_flips = flips.rolling(52).sum()
    ax.plot(roll_flips.index, roll_flips.values, lw=1.8, color=color)
    ax.axhline(roll_flips.mean(), color='black', lw=1.2, linestyle='--',
               label=f'Mean: {roll_flips.mean():.1f} flips/year')
    ax.fill_between(roll_flips.index, roll_flips.values, roll_flips.mean(),
                    where=roll_flips.values > roll_flips.mean(),
                    alpha=0.15, color=color, label='Above mean')
    ax.set_title(f'{label}: Rolling Annual Signal Flips',
                 fontsize=12, fontweight='bold')
    ax.set_ylabel('# Signal Changes in Past 52 Weeks')
    ax.legend(fontsize=10)
    ax.xaxis.set_major_locator(mdates.YearLocator(2))
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    total = flips.sum()
    print(f'{label}: {total} total flips | '
          f'{total/n_years:.1f} per year | '
          f'{total/n_weeks*100:.1f}% of weeks')

plt.suptitle('Signal Flip Frequency (driver of transaction cost differences)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


## 10. Lead/Lag Analysis

### What is Lead/Lag Analysis?

Lead/lag analysis measures the **Information Ratio as a function of the
implementation timing** between signal observation and position execution.
It is one of the most powerful robustness tools in systematic strategy research
because it tells you two critical things:

1. **Is the signal genuine or an artefact of synchronous timing?**
   A genuine signal should have its peak IR at or near lag=0.
   If IR is highest at large negative lags (signal uses future data),
   the signal is spuriously capturing forward-looking information.

2. **How persistent is the signal's forecasting power?**
   If IR remains high at lag=+1 or +2, the signal is slow-moving and
   robust to implementation delays. A sharp collapse at lag=+1 means
   the signal is time-sensitive and requires near-immediate execution.

### Important Note on Timing in This Implementation

In our **Thursday-to-Friday** implementation, the signal is already
formed from data that is one trading day old by the time of execution.
This means our lag=0 in the lead/lag chart already incorporates a
1-trading-day information lag — it is empirically realistic, not
synchronous. The lead/lag analysis therefore tests:
- **Negative lags:** Would knowing Friday's (or next week's) carry
  improve IR? (Should be 'no' — no lookahead benefit)
- **lag=0:** Our actual implementation (Thursday signal, Friday trade)
- **Positive lags:** What if we delayed execution by additional weeks?

### Methodology

We shift the Thursday-based signal by `k` weekly periods and recompute
net IR at each shift. Transaction costs are applied consistently.

```
k < 0  (lead):  holdings lead the returns — uses FUTURE weeks' signal
k = 0         : our actual Thursday-to-Friday implementation
k > 0  (lag):  uses signal from k weeks ago to trade today
```

Range tested: −10 to +10 weekly periods.

In [ ]:
# ── Lead/Lag Analysis ────────────────────────────────────────────────────
# Range of lags to test: negative = signal leads (lookahead), positive = signal lags
LAG_RANGE = range(-10, 11)   # -10 to +10 weeks

def compute_ir_at_lag(signal_col, lag, freq=FREQ):
    """
    Compute net IR when the signal is shifted by `lag` periods.

    Parameters
    ----------
    signal_col : str   Column in `sig`: 'SIGNAL_CARRY' or 'SIGNAL_CARRY_MOM'
    lag        : int   Periods to shift signal. Positive = signal used later
                       (holdings lag returns); Negative = lookahead
    freq       : int   Periods per year for annualisation

    Returns
    -------
    float  Net annualised IR at this lag
    """
    # Shift the signal: positive lag means the signal is delayed
    # shift(k) moves the signal k steps forward in time —
    # i.e. the return at time t is now paired with the signal from t-k
    shifted_signal = sig[signal_col].shift(lag)

    returns     = []
    prev_signal = None
    common_dates = fx_ret.index.intersection(shifted_signal.dropna().index)

    for date in common_dates:
        s = shifted_signal.loc[date]
        if pd.isna(s):
            prev_signal = None
            continue

        top    = top_bins.get(date, [])
        bottom = bottom_bins.get(date, [])
        if not top or not bottom:
            prev_signal = None
            continue

        n_top = len(top)
        n_bot = len(bottom)

        weights = {}
        for ccy in fx.columns:
            if ccy in top:
                weights[ccy] = s * (1.0 / n_top)
            elif ccy in bottom:
                weights[ccy] = -s * (1.0 / n_bot)
            else:
                weights[ccy] = 0.0

        gross = sum(
            weights[ccy] * fx_ret.loc[date, ccy]
            for ccy in fx.columns
        )

        # TC on signal flip (using the shifted signal)
        flipped   = (prev_signal is not None) and (s != prev_signal)
        tc_charge = 2 * TC_RATE if flipped else 0.0
        net       = gross - tc_charge

        returns.append(net)
        prev_signal = s

    r = pd.Series(returns)
    if len(r) < 20:
        return np.nan
    ann_ret = r.mean() * freq
    ann_vol = r.std()  * np.sqrt(freq)
    return ann_ret / ann_vol if ann_vol != 0 else np.nan


# Compute IR at each lag for both signals
print('Computing lead/lag IR profile...')
ll_results = {}
for sig_label, sig_col in [
    ('CARRY',     'SIGNAL_CARRY'),
    ('CARRY-MOM', 'SIGNAL_CARRY_MOM'),
]:
    irs = {}
    for lag in LAG_RANGE:
        irs[lag] = compute_ir_at_lag(sig_col, lag)
    ll_results[sig_label] = irs
    print(f'  {sig_label}: lag=0 IR = {irs[0]:.3f}, '
        f'lag=+1 IR = {irs[1]:.3f}, '
        f'lag=+2 IR = {irs[2]:.3f}')

print('Done.')


### Lead/Lag Chart — How to Read It

**Light blue bars (left of zero):** Un-attainable. These use future signal
information and serve only as a theoretical upper bound. If these were
dramatically higher than the red bar, it would indicate lookahead bias.

**Red bar (lag=0):** Our actual Thursday-to-Friday implementation.
For CARRY: IR = **0.359**. For CARRY-MOM: IR = **1.081**.

**Orange/blue bars (right of zero):** Attainable but suboptimal —
what IR you would get if you delayed execution by k additional weeks.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, sig_label, color in [
    (axes[0], 'CARRY',     'steelblue'),
    (axes[1], 'CARRY-MOM', 'darkorange'),
]:
    lags = list(ll_results[sig_label].keys())
    irs  = [ll_results[sig_label][l] for l in lags]
    ir0  = ll_results[sig_label][0]

    # Colour bars: negative lag = lighter (unattainable), zero = red, positive = solid
    bar_colors = []
    for l in lags:
        if l < 0:
            bar_colors.append('#aec6e8')   # light blue — lookahead (unattainable)
        elif l == 0:
            bar_colors.append('#d62728')   # red — actual implementation
        else:
            bar_colors.append(color)       # solid — lagged (attainable)

    bars = ax.bar(lags, irs, color=bar_colors, edgecolor='white', linewidth=0.5)

    # Highlight lag=0 and lag=+1
    ax.axvline(0, color='black', lw=1.2, linestyle='--', alpha=0.7)
    ax.axvline(1, color='gray',  lw=1.0, linestyle=':', alpha=0.6)
    ax.axhline(0, color='black', lw=0.8)

    # Annotate key bars
    for lag_key, label_str in [(0, 'Actual\n(lag=0)'), (1, 'lag=+1')]:
        if lag_key in ll_results[sig_label]:
            ir_val = ll_results[sig_label][lag_key]
            ax.text(lag_key, ir_val + 0.015, f'{ir_val:.3f}',
                    ha='center', va='bottom', fontsize=9,
                    fontweight='bold',
                    color='darkred' if lag_key == 0 else 'dimgray')

    # Legend patches
    from matplotlib.patches import Patch
    legend_elements = [
        Patch(facecolor='#aec6e8', label='Negative lag (lookahead — un-attainable)'),
        Patch(facecolor='#d62728', label='Lag = 0 (our actual implementation)'),
        Patch(facecolor=color,     label='Positive lag (implementation delay)'),
    ]
    ax.legend(handles=legend_elements, fontsize=8, loc='upper right')

    ax.set_title(f'{sig_label}: Net IR by Lead/Lag Period',
                 fontsize=13, fontweight='bold')
    ax.set_xlabel('Lead/Lag (weeks)  [negative = signal uses future data]')
    ax.set_ylabel('Net Information Ratio')
    ax.set_xticks(lags)
    ax.set_xticklabels(lags, fontsize=9)

plt.suptitle(
    'Lead/Lag Analysis: Net IR vs Signal Implementation Timing\n'
    'Red bar = our actual strategy (lag=0). Light blue = un-attainable (lookahead).',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()

# Print summary table
print('Lead/Lag IR Summary:')
print(f'  {"Lag":<6} {"CARRY IR":<14} {"CARRY-MOM IR"}')
print('  ' + '-'*36)
for lag in LAG_RANGE:
    carry_ir = ll_results['CARRY'][lag]
    mom_ir   = ll_results['CARRY-MOM'][lag]
    flag = ' <- actual' if lag == 0 else (' <- best attainable' if lag == 1 else '')
    print(f'  {lag:<6} {carry_ir:<14.3f} {mom_ir:.3f}{flag}')


### Lead/Lag Interpretation

**CARRY signal — flat and robust:**
The IR is remarkably stable across all lags (range: 0.24–0.49).
The lag=0 IR of 0.359 barely changes at lag=+1 (0.349) — only a 2.8%
reduction. The negative-lag bars are not materially higher than lag=0,
confirming there is **no lookahead bias**. The carry *level* is a slow-moving,
persistent signal: whether you observe it one week early or one week late
makes almost no difference. This also confirms that Thursday prices are
a valid proxy for the regime signal — the day difference from Friday is
economically irrelevant for a slow-moving level signal.

**CARRY-MOM signal — sharp spike at lag=0:**
The IR peaks sharply at lag=0 (1.081) and collapses to 0.015 at lag=+1.
This is the most important finding in the robustness section.

*Does this indicate lookahead bias?* **No.** Critically, the negative-lag
bars are also near zero or negative — using future signal data does NOT
improve performance. The pattern is a **symmetric spike at lag=0**: the
information is only useful at the exact Thursday observation moment.
This is the signature of a genuine contemporaneous signal, not a
lookahead artefact.

*Why does the spike occur?* CARRY-MOM fires at a crossing event — when
carry crosses its 4-week MA. One week later, the crossing has passed:
either the regime has shifted (position correct, no new signal) or the
carry has crossed back (position now wrong). In either case, acting on
a week-old MOM signal is uninformative or harmful. This is not unique to
our strategy — it is a known property of moving-average crossover signals.

*Practical implication:* CARRY-MOM requires Thursday-to-Friday (or
equivalent same-week) execution to capture its IR. A full week's delay
destroys the signal. Since all 8 currencies trade on liquid electronic
markets where Friday close execution is straightforward, this is not a
practical barrier but must be acknowledged as a constraint.

## 11. Jackknife Robustness Test

### What is a Jackknife Test?

The Jackknife test (also called 'leave-one-out' analysis) evaluates whether
the strategy's performance is **genuinely diversified** across assets or
**over-reliant on a few key currencies**.

**Methodology:**
1. Start with the full 8-currency universe → compute the baseline IR
2. Iteratively remove one currency at a time from the universe
3. Re-run the full strategy without that currency (remaining currencies
   in the same bin are equally reweighted — a like-for-like comparison)
4. Compute: **Contribution to IR = IR_all − IR_without_that_currency**

**Interpretation of the contribution score:**

| Contribution | Meaning |
|---|---|
| Large positive (>0.10) | Currency is a strong contributor — removing it significantly hurts IR |
| Near zero (±0.05) | Currency is neutral — strategy works equally well without it |
| Negative (<0) | Currency is a **detractor** — removing it actually *improves* IR |

A well-diversified strategy should show **small, roughly equal contributions**.
A strategy with one dominant contributor is fragile — it is effectively
a single-asset bet dressed up as a diversified portfolio.

In [ ]:
def run_jackknife_backtest(signal_col, excluded_ccy, freq=FREQ):
    """
    Run the strategy with one currency removed from the universe.

    Parameters
    ----------
    signal_col    : str  'SIGNAL_CARRY' or 'SIGNAL_CARRY_MOM'
    excluded_ccy  : str  Currency ticker to exclude (e.g. 'NOK')
    freq          : int  Periods per year

    Returns
    -------
    float  Net annualised IR with that currency excluded
    """
    returns     = []
    prev_signal = None
    common_dates = fx_ret.index.intersection(sig.index)

    for date in common_dates:
        s      = sig.loc[date, signal_col]
        top    = [c for c in top_bins[date]    if c != excluded_ccy]
        bottom = [c for c in bottom_bins[date] if c != excluded_ccy]

        # Skip weeks where the excluded currency was the only one in a bin
        if not top or not bottom:
            prev_signal = None
            continue

        n_top = len(top)
        n_bot = len(bottom)

        weights = {}
        for ccy in fx.columns:
            if ccy == excluded_ccy:
                weights[ccy] = 0.0
            elif ccy in top:
                weights[ccy] = s * (1.0 / n_top)
            elif ccy in bottom:
                weights[ccy] = -s * (1.0 / n_bot)
            else:
                weights[ccy] = 0.0

        gross = sum(
            weights[ccy] * fx_ret.loc[date, ccy]
            for ccy in fx.columns
        )

        flipped   = (prev_signal is not None) and (s != prev_signal)
        tc_charge = 2 * TC_RATE if flipped else 0.0
        net       = gross - tc_charge

        returns.append(net)
        prev_signal = s

    r = pd.Series(returns)
    if len(r) < 20:
        return np.nan
    ann_ret = r.mean() * freq
    ann_vol = r.std()  * np.sqrt(freq)
    return ann_ret / ann_vol if ann_vol != 0 else np.nan


# ── Run Jackknife ─────────────────────────────────────────────────────────
print('Running Jackknife test...')

# Baseline IRs (full universe, net)
ir_carry_full = compute_metrics(res_carry['net_return'])['Information Ratio']
ir_mom_full   = compute_metrics(res_mom['net_return'])['Information Ratio']

jk_results = {}
for sig_label, sig_col, ir_full in [
    ('CARRY',     'SIGNAL_CARRY',     ir_carry_full),
    ('CARRY-MOM', 'SIGNAL_CARRY_MOM', ir_mom_full),
]:
    contributions = {}
    for ccy in fx.columns:
        ir_without = run_jackknife_backtest(sig_col, ccy)
        contributions[ccy] = ir_full - ir_without  # positive = currency adds value
        print(f'  {sig_label} | Remove {ccy}: IR drops from '
              f'{ir_full:.3f} to {ir_without:.3f} '
              f'(contribution: {contributions[ccy]:+.3f})')
    jk_results[sig_label] = contributions

print('\nJackknife complete.')


### Jackknife Visualisation — How to Read It

Each bar represents one currency being removed. The bar height equals
`IR_all − IR_without_that_currency`:

- **Tall positive bar (blue):** This currency is important — its removal
  substantially reduces the IR. The strategy relies on this currency.
- **Near-zero bar:** This currency is neutral — the strategy performs
  almost identically without it.
- **Negative bar (salmon):** This currency is a detractor — the strategy
  actually performs *better* without it.

The label on each bar shows both the contribution score (+/-) and the
IR that would result if that currency were excluded.

Bars are sorted from highest to lowest contribution for easy identification
of the most and least important currencies.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, sig_label, ir_full, color in [
    (axes[0], 'CARRY',     ir_carry_full, 'steelblue'),
    (axes[1], 'CARRY-MOM', ir_mom_full,   'darkorange'),
]:
    contribs = jk_results[sig_label]
    # Sort by contribution descending
    sorted_items = sorted(contribs.items(), key=lambda x: x[1], reverse=True)
    ccys_sorted  = [c for c, _ in sorted_items]
    vals_sorted  = [v for _, v in sorted_items]

    bar_colors = ['steelblue' if v >= 0 else 'salmon' for v in vals_sorted]
    bars = ax.bar(range(len(ccys_sorted)), vals_sorted,
                  color=bar_colors, edgecolor='white', linewidth=0.5)

    ax.axhline(0, color='black', lw=0.9)

    # Annotate with IR-without value
    ir_without_vals = {c: ir_full - contribs[c] for c in ccys_sorted}
    for i, (ccy, v) in enumerate(zip(ccys_sorted, vals_sorted)):
        ir_wo = ir_without_vals[ccy]
        ypos  = v + 0.008 if v >= 0 else v - 0.025
        ax.text(i, ypos,
                f'{v:+.3f}\n(IR w/o: {ir_wo:.3f})',
                ha='center', va='bottom', fontsize=8.5)

    ax.set_xticks(range(len(ccys_sorted)))
    ax.set_xticklabels(ccys_sorted, fontsize=11)
    ax.set_title(
        f'{sig_label}: Jackknife Contribution to Net IR\n'
        f'(Baseline full-universe IR = {ir_full:.3f})',
        fontsize=12, fontweight='bold'
    )
    ax.set_xlabel('Currency removed')
    ax.set_ylabel('IR_all  −  IR_without_currency')

    # Add annotations explaining bar direction
    # ax.text(0.01, 0.97, 'Positive = contributor (removing hurts IR)',
    #         transform=ax.transAxes, fontsize=8,
    #         va='top', color='steelblue')
    # ax.text(0.01, 0.91, 'Negative = detractor (removing improves IR)',
    #         transform=ax.transAxes, fontsize=8,
    #         va='top', color='salmon')

plt.suptitle(
    'Jackknife Test: IR Contribution of Each Currency',
    fontsize=13, fontweight='bold'
)
plt.tight_layout()
plt.show()


In [ ]:
# ── Jackknife detail table ────────────────────────────────────────────────
jk_table = pd.DataFrame({
    'CARRY IR w/o':   {c: ir_carry_full - jk_results['CARRY'][c]
                       for c in fx.columns},
    'CARRY contrib':  jk_results['CARRY'],
    'MOM IR w/o':     {c: ir_mom_full - jk_results['CARRY-MOM'][c]
                       for c in fx.columns},
    'MOM contrib':    jk_results['CARRY-MOM'],
}).round(3)

jk_table.index.name = 'Removed Currency'
print(f'Jackknife detail table (baseline: CARRY IR={ir_carry_full:.3f}, '
      f'MOM IR={ir_mom_full:.3f}):')
display(jk_table.sort_values('MOM contrib', ascending=False))


### Jackknife Interpretation — Actual Results

**CARRY-MOM (baseline IR = 1.081):**

| Currency | IR w/o | Contribution | Role |
|---|---|---|---|
| **KRW** | 0.904 | **+0.177** | Strongest contributor — critical to short leg |
| **NOK** | 0.930 | **+0.151** | Second strongest — anchor of top bin |
| **COP** | 0.951 | +0.130 | Meaningful top-bin diversification |
| **JPY** | 0.967 | +0.114 | Safe-haven short adds genuine value |
| **INR** | 1.048 | +0.033 | Modest positive contribution |
| **CAD** | 1.052 | +0.029 | Near-neutral |
| **ZAR** | 1.083 | −0.002 | Essentially neutral (near zero) |
| **MXN** | 1.095 | **−0.014** | Slight detractor |

**Key findings:**

**The strategy is more diversified in the Thursday version** than in the
Friday version. The top-4 contributions (KRW, NOK, COP, JPY) are
more evenly spread (0.177, 0.151, 0.130, 0.114) compared to the prior
version's concentration in KRW and NOK. This reflects the Thursday signal
capturing slightly different regime entry points that benefit additional currencies.

**KRW and NOK remain the two most important currencies.** KRW (Korean Won)
is the most reliable short — Korea imports virtually all its oil, making
KRW highly responsive to oil regime changes. NOK is the most reliable
long — Norway's economy is the most oil-export-dependent in the G10.

**COP adds genuine value (+0.130).** This is a meaningful contribution
for a currency that appeared in the strategy through an NDF and could
have been a detractor given its structural depreciation. COP's oil-exporter
beta clearly outweighs its depreciation drag, validating the decision
to include it in the top bin.

**JPY (+0.114) is more important in the Thursday version.** As a safe-haven
currency, JPY tends to appreciate sharply during risk-off episodes —
exactly when oil prices fall and the contango signal fires. Being short
JPY in backwardation and long JPY in contango captures this safe-haven
dynamic effectively.

**MXN is a slight detractor (−0.014).** Its bin rotation (top bin 2007–2010,
then back as bottom-bin from 2020–2022) creates two-sided noise. The strategy
would marginally improve without MXN, but the difference is economically
negligible and the theoretical case for including MXN as a major oil
producer remains valid.

**Overall assessment:** The Jackknife test reveals a well-diversified strategy.
No single currency has a contribution above 0.20 (the threshold for
dangerous concentration), and the contributions are spread across both
bins (exporters and importers). This confirms the multi-currency design
is genuinely adding diversification value rather than just diluting a
single-pair trade.

## 12. Summary & Conclusions

### Key Findings

**1. The oil carry signal contains genuine information about FX returns.**
The backwardation/contango regime of the Brent futures curve systematically
predicts the relative performance of oil-exporter vs oil-importer currencies.
This is consistent with Bouchouev & Zuo (2020) and the broader commodity
currency literature.

**2. CARRY-MOM significantly outperforms CARRY.**
The momentum overlay — positioning based on whether Thursday's carry is
improving or deteriorating relative to its 4-week average — adds substantial
value beyond the raw carry level signal. The gross IR improvement from
0.379 to 1.284 reflects the overlay's ability to anticipate regime
transitions before they fully manifest and to profit from both regimes
rather than just one.

**3. The Thursday-to-Friday implementation is empirically sound.**
Using Thursday's carry observation to trade at Friday's close provides a
full trading day between signal formation and execution. The CARRY signal's
agreement rate of 98.2% with the Friday version confirms that the one-day
shift does not materially alter the regime calls. CARRY-MOM's 88.6%
agreement rate reflects genuine additional information content at regime
boundaries — the Thursday observation sometimes captures regime transitions
one day earlier.

**4. Transaction costs matter but do not eliminate CARRY-MOM's edge.**
At 0.05% base-case, CARRY-MOM's return declines from 10.41% to 8.76%
and IR from 1.284 to 1.081. The strategy remains strongly profitable.
The breakeven transaction cost is well above 0.20% per trade.

**5. COP (Colombia NDF) adds genuine diversification value.**
The Jackknife contribution of +0.130 for COP (CARRY-MOM) confirms that
the Colombian NDF adds real signal beyond the Norwegian and Canadian
currencies, validating its inclusion despite the OTC market friction.

**6. The Lead/Lag analysis confirms no lookahead bias.**
The symmetric spike at lag=0 (with negative lags showing no benefit)
confirms that CARRY-MOM's high IR at lag=0 reflects genuine signal
quality, not synchronous observation-execution bias. The strategy does
require timely execution (within the same week as signal observation)
but this is feasible given the liquidity of the instruments.

---

### Strategy Scorecard (Net of 0.05% Transaction Costs)

| Metric | CARRY | CARRY-MOM | Assessment |
|---|---|---|---|
| Ann. Net Return | 2.95% | **8.76%** | MOM >> CARRY |
| Information Ratio | 0.359 | **1.081** | MOM investable; CARRY marginal |
| Sortino Ratio | 0.602 | **1.830** | MOM excellent risk-adjusted |
| Max Drawdown | −16.9% | −15.4% | Comparable; CARRY-MOM better gross |
| Positive years | 8/19 | 14/19 | MOM more consistent |
| Annual flips | 1.7 | 16.5 | MOM more costly to trade |

---

### Limitations & Directions for Future Work

- **Bin assignments use contemporaneous export data.** Using prior-year
  bins would be strictly more realistic — annual oil trade statistics are
  published with a lag after year-end.

- **Equal weighting ignores volatility differences.** A risk-parity
  (inverse-volatility) weighting scheme would equalise risk contributions
  across currencies, potentially improving the IR further.

- **The COP NDF introduces OTC market friction** not fully captured by
  the fixed 0.05% cost assumption. A higher cost for COP specifically
  could be applied if granular bid-offer data becomes available.

- **WTI (CL3/CL12) as an alternative signal** could be tested using the
  Signal Construction notebook provided alongside this file, to assess
  whether the Brent-based signal generalises to the WTI term structure.

## 13. Save Results

Saves weekly return series, annual breakdowns, and overall performance
summary for further analysis or presentation. All files reflect the
Thursday-to-Friday signal implementation with 0.05% transaction costs.

In [ ]:
# Weekly results
res_carry[['gross_return','tc_charge','net_return',
           'signal','flip','top_bin','bottom_bin']].to_csv(
    'Final_CARRY_weekly.csv')
res_mom[['gross_return','tc_charge','net_return',
         'signal','flip','top_bin','bottom_bin']].to_csv(
    'Final_MOM_weekly.csv')

# Annual breakdowns
ann_carry.to_csv('Final_CARRY_annual.csv')
ann_mom.to_csv('Final_MOM_annual.csv')

# Overall metrics
metrics.to_csv('Final_metrics_summary.csv')

print('Files saved:')
for f in ['Final_CARRY_weekly.csv', 'Final_MOM_weekly.csv',
          'Final_CARRY_annual.csv', 'Final_MOM_annual.csv',
          'Final_metrics_summary.csv']:
    print(f'  {f}')

print('\nFinal performance summary (net of transaction costs):')
display(metrics[['CARRY  Net','MOM  Net']])
